<a href="https://colab.research.google.com/github/velascocafe23/nlp-maestria-upb/blob/main/03-transformers-semana2/lab_transformers_semana2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Entrega 3 · Lab Transformers · Semana 2 (embeddings contextuales, comparativa multi-modelo, atención, clustering, zero-shot)** — Sebastián Velasco Ardila · Maestría en Ciencia de Datos, UPB · Curso de NLP

Notebook guiado, ejecutado completo en T4, incluido el bonus de zero-shot. Las reflexiones están en `ENTREGA.md`.

# Embeddings Contextuales y Transformers en Español
**Curso de Maestría en NLP (UPB Medellín) - Semana 2 - Viernes**

| | |
|---|---|
| **Sesión** | Semana 2, Viernes |
| **Duración estimada** | 90 minutos |
| **Entorno** | Google Colab gratuito con **GPU T4** (Entorno de ejecución -> Cambiar tipo de entorno -> T4 GPU) |
| **Dominio** | Banca y finanzas colombianas |

### Instrucciones para el instructor
1. **Antes de empezar:** pedir a los estudiantes activar la GPU en **Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> T4 GPU**. La celda de configuración lo verifica.
2. Compartir el link de Colab al iniciar el módulo.
3. La primera ejecución descarga los modelos desde HuggingFace (unos minutos); avisar a los estudiantes.
4. Pausar en cada `# === PAUSA: DISCUSIÓN ===` para discutir en grupo (3-5 min).
5. Guardar copia: **Archivo -> Guardar una copia en Drive**.

### Qué vas a aprender
- Por qué los embeddings **estáticos** (GloVe) no capturan el contexto y dónde falla eso.
- Cómo usar **HuggingFace pipelines** para análisis de sentimiento y NER en español, sin entrenar nada.
- Qué son los embeddings **contextuales** que produce un Transformer y por qué resuelven la ambigüedad.
- Cómo **comparar varios modelos** en la misma tarea y elegir con criterio (no siempre gana el más grande).
- Cómo **visualizar la atención** de un Transformer para entender qué está "mirando".
- Todo con ejemplos de **banca y finanzas colombianas** (Bancolombia, Banco de la República, etc.).

## Configuración del entorno


In [ ]:
# @title Configuración - instalar librerías y verificar GPU
# transformers + torch para los modelos; gensim para los embeddings estáticos de GloVe.
!pip install -q transformers torch gensim

import warnings
warnings.filterwarnings('ignore')
import numpy as np
import torch

# Verificamos que la GPU T4 esté activa (Entorno de ejecución -> T4 GPU).
if torch.cuda.is_available():
    print('GPU activa:', torch.cuda.get_device_name(0))
    DEVICE = 0          # los pipelines de HuggingFace usan 0 = primera GPU
else:
    print('ATENCION: no hay GPU. Ve a Entorno de ejecucion -> Cambiar tipo -> T4 GPU.')
    print('El lab funciona igual en CPU, solo mas lento.')
    DEVICE = -1         # -1 = CPU

print('Entorno listo.')


---
## Paso 1: El problema de la ambigüedad — motivación

En labs anteriores representamos texto con **Bag of Words** y **TF-IDF**: cada palabra es una
posición fija en un vector, sin ningún conocimiento de significado. Word2Vec y GloVe dieron un
paso adelante: aprenden un **vector por palabra** que captura similitud semántica
(`rey - hombre + mujer ≈ reina`).

#### El problema: los embeddings estáticos dan UN solo vector por palabra
La palabra **"banco"** significa cosas distintas según el contexto:
- *"Fui al **banco** a depositar plata"* → institución financiera (Bancolombia, Davivienda...).
- *"Me senté en el **banco** del parque"* → un asiento.

Con GloVe, ambos usos comparten **exactamente el mismo vector**. El modelo no puede distinguir los
dos sentidos, porque el vector se calculó una sola vez, sin contexto.

#### La solución (Paso 3): embeddings contextuales
Los **Transformers** (BERT, RoBERTa, BETO...) producen un vector **distinto para cada aparición**
de la palabra, en función de las palabras que la rodean. Ese es el salto conceptual de este lab.

> **Nota:** usamos GloVe en **inglés** (`glove-wiki-gigaword-100`) porque los vectores GloVe
> pre-entrenados más comunes son en inglés. La palabra ambigua equivalente es **"bank"**
> (banco financiero / orilla de río). El concepto es idéntico para el español.


In [ ]:
# @title Paso 1: Cargar GloVe — un solo vector por palabra (sin contexto)
import gensim.downloader as api

print('Descargando GloVe (glove-wiki-gigaword-100)... puede tardar la primera vez.')
glove = api.load('glove-wiki-gigaword-100')
print('GloVe cargado. Vocabulario:', len(glove.index_to_key), 'palabras')

# El vector de 'bank' es UNO solo, sin importar el contexto.
vec_bank = glove['bank']
print('\nShape del vector de "bank":', vec_bank.shape)
print('Primeras 5 dimensiones:', np.round(vec_bank[:5], 3))

# Dos oraciones con 'bank' en sentidos distintos:
#   "I went to the bank to deposit money"  -> banco financiero
#   "I sat on the bank of the river"        -> orilla del rio
# Como GloVe es estatico, el vector de 'bank' es IDENTICO en ambas:
vec_bank_financiero = glove['bank']   # "...bank to deposit money"
vec_bank_rio        = glove['bank']   # "...bank of the river"

iguales = np.array_equal(vec_bank_financiero, vec_bank_rio)
print(f'\n¿El vector de "bank" es identico en ambas oraciones? {iguales}')


In [ ]:
# @title Paso 1 (visual): la similitud coseno es EXACTAMENTE 1.0 (mismo vector)
import matplotlib.pyplot as plt

def cos_sim(a, b):
    a = a / np.linalg.norm(a)
    b = b / np.linalg.norm(b)
    return float(np.dot(a, b))

sim_glove = cos_sim(vec_bank_financiero, vec_bank_rio)
print(f'Similitud coseno entre "bank" (oracion financiera) y "bank" (oracion del rio): {sim_glove:.3f}')
print('=> Es 1.000 porque GloVe usa SIEMPRE el mismo vector. No hay desambiguacion por contexto.')

# Grafico de barra unica que deja claro el 1.0
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(['GloVe: bank vs bank\n(dos contextos)'], [sim_glove], color='#1B3A6B', alpha=0.85, width=0.5)
ax.axhline(1.0, color='#E8821A', linestyle='--', linewidth=1)
ax.text(0, sim_glove + 0.02, f'{sim_glove:.3f}', ha='center', fontweight='bold')
ax.set_ylim(0, 1.15)
ax.set_ylabel('Similitud coseno')
ax.set_title('Embedding estatico: mismo vector => similitud = 1.0', fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Ademas: como es estatico, su similitud con otras palabras tampoco cambia con el contexto.
financieras = ['money', 'credit', 'loan']
lugar       = ['river', 'park', 'bench']
print('\nSimilitud de "bank" (vector unico) con cada sentido:')
print('  Sentido FINANCIERO:')
for w in financieras:
    if w in glove:
        print(f'    bank ~ {w:8s}: {glove.similarity("bank", w):.3f}')
print('  Sentido LUGAR/ORILLA:')
for w in lugar:
    if w in glove:
        print(f'    bank ~ {w:8s}: {glove.similarity("bank", w):.3f}')
print('\nEstas similitudes NUNCA cambian, aparezca "bank" donde aparezca.')


### Reflexión 1 - Embeddings estáticos vs. contextuales

> 1. ¿Por qué un único vector por palabra es un problema para palabras polisémicas como "banco", "cuenta", "interés" o "acción"? (todas muy comunes en finanzas)
> 2. ¿Qué información necesita un modelo para asignar el sentido correcto a "banco"?
> 3. Word2Vec/GloVe resuelven analogías (`rey-hombre+mujer≈reina`). ¿Por qué eso no basta para la ambigüedad de sentido?

# === PAUSA: DISCUSIÓN ===


---
## Paso 2: HuggingFace Pipelines — inferencia en 3 líneas

#### ¿Qué es HuggingFace?
**HuggingFace** es, en la práctica, el "GitHub de los modelos de IA": un repositorio público con
cientos de miles de modelos pre-entrenados (muchos en español) que puedes descargar y usar gratis.
En lugar de entrenar un Transformer desde cero (miles de dólares en GPU), reutilizamos uno que ya
aprendió el idioma.

#### ¿Qué es un pipeline?
Un **pipeline** de la librería `transformers` encapsula los 3 pasos de la inferencia en una sola
llamada: (1) **tokeniza** el texto, (2) lo pasa por el **modelo** Transformer, (3) **formatea** la
salida en un resultado legible (etiqueta + score). Esto resuelve una tarea en **3 líneas**.

Pasamos `device=DEVICE` para que el pipeline use la **GPU T4** si está disponible.


### 2a: Análisis de sentimiento en español (tweets financieros)

Usamos **`pysentimiento/robertuito-sentiment-analysis`**, un modelo RoBERTa entrenado sobre
**tweets en español**. Devuelve las etiquetas `POS` (positivo), `NEU` (neutro) y `NEG` (negativo),
junto con la probabilidad de cada clase.


In [ ]:
# @title Paso 2a: Sentimiento en español con un pipeline de HuggingFace
from transformers import pipeline

# 3 lineas: cargamos el pipeline con un modelo de sentimiento en español (tweets).
# return_all_scores=True nos da la probabilidad de CADA clase (POS/NEU/NEG).
clf = pipeline('sentiment-analysis',
               model='pysentimiento/robertuito-sentiment-analysis',
               top_k=None,           # devuelve el score de todas las clases
               device=DEVICE)

# 10 tweets variados sobre banca/finanzas colombianas (positivos, negativos, neutros).
tweets = [
    'Bancolombia me acaba de aprobar el credito de vivienda, feliz con la atencion!',   # pos
    'Llevo 40 minutos en la fila del banco y no avanza nada, pesimo servicio',          # neg
    'El Banco de la Republica mantuvo la tasa de interes en la reunion de hoy',          # neu
    'La app de Nequi lleva caida toda la manana, no puedo pagar nada, que rabia',        # neg
    'Excelente la nueva tarjeta de Davivienda, cashback en todas las compras',           # pos
    'Segun el DANE la inflacion de este mes fue del 0.3 por ciento',                     # neu
    'Me cobraron una cuota de manejo que no autorice, voy a reclamar ya mismo',          # neg
    'Daviplata me salvo el dia, transferencia inmediata y sin filas, lo recomiendo',     # pos
    'La accion de Ecopetrol cerro estable en la Bolsa de Valores de Colombia',           # neu
    'Gracias al asesor de Bancolombia que me ayudo a reestructurar la deuda, mil gracias', # pos
]

MAP_SENT = {'NEG': 'negativo', 'NEU': 'neutro', 'POS': 'positivo'}

print('SENTIMIENTO PREDICHO POR EL TRANSFORMER (con probabilidad por clase)')
print('=' * 78)
for tw in tweets:
    scores = clf(tw)[0]                       # lista de dicts {label, score}
    top = max(scores, key=lambda d: d['score'])
    etiqueta = MAP_SENT.get(top['label'], top['label'])
    # Probabilidades de todas las clases, ordenadas para mostrarlas
    detalle = '  '.join(f'{MAP_SENT.get(s["label"], s["label"])[:3]}={s["score"]:.2f}'
                        for s in sorted(scores, key=lambda d: d['label']))
    print(f'\n[{etiqueta.upper():9s}]  {tw[:64]}')
    print(f'            probabilidades -> {detalle}')


#### Comparación con el lab de clasificación con BoW (BoW/TF-IDF + Regresión Logística)

En el laboratorio anterior clasificamos sentimiento con **Bag of Words / TF-IDF + Regresión Logística**. Ese
enfoque clásico:
- Trata cada palabra como una posición independiente: **"no lo recomiendo"** y **"lo recomiendo"**
  comparten casi todas las palabras, así que le cuesta la **negación**.
- Necesita **preprocesamiento** (limpiar, quitar stopwords, a veces stemming) y **entrenar** con
  datos etiquetados.
- Es rápido e **interpretable** (puedes ver qué palabras pesan).

El Transformer que acabas de usar, en cambio:
- **No requiere entrenamiento** ni preprocesamiento: funciona directo sobre el tweet crudo.
- Entiende **contexto** y **negación** ("no puedo pagar", "no autorice"), que es justo donde BoW
  falla.
- Suele lograr más exactitud en texto informal (jerga, emojis, ironía), típico de redes sociales.

**¿Cuándo importa la diferencia?** En tweets cortos, informales y con negación (como los de arriba).
**¿Cuándo NO tanto?** En clasificación temática con vocabulario muy marcado (deporte vs. finanzas),
donde BoW/TF-IDF ya es fuerte, más rápido y más fácil de explicar.


### Reflexión 2a - Transformer vs. modelo clásico

> 1. Mira los tweets con negación ("no puedo pagar", "no autorice"). ¿Por qué BoW/TF-IDF del laboratorio anterior fallaría ahí y el Transformer no?
> 2. El Transformer es más lento y menos interpretable. ¿En qué escenario bancario preferirías igualmente TF-IDF + LogReg?
> 3. ¿Qué riesgo hay en confiar en un modelo pre-entrenado (entrenado con tweets genéricos) para analizar los mensajes de TU banco sin validarlo con tus propios datos?

# === PAUSA: DISCUSIÓN ===


### 2b: Reconocimiento de entidades (NER) en español

El **NER (Named Entity Recognition)** detecta y clasifica entidades dentro del texto:
**PER** (personas), **ORG** (organizaciones), **LOC** (lugares) y **MISC** (misceláneas). Es clave
para extraer información estructurada de noticias, contratos o reportes financieros.

Usamos **`mrm8488/bert-spanish-cased-finetuned-ner`** (BETO afinado para NER en español). El
parámetro `aggregation_strategy='simple'` agrupa los sub-tokens en entidades completas (p. ej. une
"Banco" + "##lombia" en la entidad `Bancolombia`).


In [ ]:
# @title Paso 2b: NER en español sobre noticias financieras colombianas
ner = pipeline('ner',
               model='mrm8488/bert-spanish-cased-finetuned-ner',
               aggregation_strategy='simple',
               device=DEVICE)

# Noticias financieras colombianas reales (parafraseadas). Al menos 3 oraciones.
noticias = [
    'Bancolombia anuncio nuevas inversiones en Medellin durante el ultimo trimestre.',
    'El Banco de la Republica, dirigido por Leonardo Villar, mantuvo la tasa de interes en Bogota.',
    'La Bolsa de Valores de Colombia cerro al alza tras el reporte de Grupo Nutresa y Ecopetrol.',
    'Avianca firmo un acuerdo con Airbus para renovar su flota en America Latina.',
    'Davivienda y Nequi ampliaron sus servicios de pagos digitales en Colombia.',
]

print('ENTIDADES DETECTADAS (NER)')
print('=' * 78)
for texto in noticias:
    print(f'\nNoticia: {texto}')
    entidades = ner(texto)
    if not entidades:
        print('  (sin entidades detectadas)')
    for e in entidades:
        grupo = e.get('entity_group', e.get('entity', '?'))
        print(f'  - {e["word"]:28s} -> {grupo:5s} (score {e["score"]:.2f})')


In [ ]:
# @title Paso 2b (visual): conteo de entidades por tipo
from collections import Counter

conteo_tipos = Counter()
for texto in noticias:
    for e in ner(texto):
        grupo = e.get('entity_group', e.get('entity', 'MISC'))
        conteo_tipos[grupo] += 1

if conteo_tipos:
    tipos   = list(conteo_tipos.keys())
    valores = [conteo_tipos[t] for t in tipos]
    colores = ['#1B3A6B', '#0EA5A0', '#16A34A', '#E8821A', '#7F8C8D']
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(tipos, valores, color=colores[:len(tipos)], alpha=0.85)
    for i, v in enumerate(valores):
        ax.text(i, v + 0.05, str(v), ha='center', fontweight='bold')
    ax.set_ylabel('Numero de entidades')
    ax.set_title('Entidades detectadas por tipo (noticias financieras)', fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('No se detectaron entidades.')

print('Tipos: PER=persona, ORG=organizacion, LOC=lugar, MISC=miscelanea')


### Reflexión 2b - NER

> 1. ¿Qué entidades detectó bien y cuáles confundió o partió mal? ¿Por qué es difícil el caso de "Banco de la República" (¿ORG o LOC?)?
> 2. ¿Cómo usarías NER en un producto financiero real (p. ej. leer contratos, monitorear noticias de la competencia, detectar menciones de clientes)?
> 3. ¿Por qué un enfoque de Bag of Words no serviría para esta tarea de extracción de entidades?

# === PAUSA: DISCUSIÓN ===


---
## Paso 3: Embeddings contextuales con BETO — el momento "¡ajá!"

En el Paso 1 vimos que GloVe le da a "banco"/"bank" **un único vector**, con similitud consigo
mismo de **1.000**, sin importar el contexto. Ahora cerramos el círculo: **BETO** (BERT entrenado
en español, `dccuchile/bert-base-spanish-wwm-cased`) produce un vector **distinto para cada
aparición** de la palabra, según la oración completa.

Tomamos la MISMA palabra **"banco"** en dos oraciones colombianas con sentidos diferentes y
comparamos sus vectores:
- *"Fui al banco a sacar plata"* → institución financiera.
- *"Me senté en el banco del parque"* → asiento.

Si el modelo es realmente contextual, los dos vectores **no** serán idénticos: la similitud coseno
será **< 1.0** (a diferencia del 1.000 de GloVe).


In [ ]:
# @title Paso 3: BETO da un vector distinto a 'banco' segun el contexto
from transformers import AutoTokenizer, AutoModel

BETO = 'dccuchile/bert-base-spanish-wwm-cased'
tokenizer   = AutoTokenizer.from_pretrained(BETO)
modelo_beto = AutoModel.from_pretrained(BETO)
modelo_beto.eval()

# Mover el modelo a GPU si esta disponible.
torch_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
modelo_beto.to(torch_device)
print('BETO cargado en:', torch_device)

def embedding_de_palabra(oracion, palabra='banco'):
    """Devuelve el vector contextual (768 dim) de `palabra` dentro de `oracion`."""
    enc = tokenizer(oracion, return_tensors='pt').to(torch_device)
    with torch.no_grad():
        salida = modelo_beto(**enc).last_hidden_state[0]   # (n_tokens, 768)
    tokens = tokenizer.convert_ids_to_tokens(enc['input_ids'][0])
    # Primer sub-token que contenga la palabra objetivo.
    idx = next(i for i, t in enumerate(tokens) if palabra in t.lower())
    return salida[idx].cpu()

oracion_fin    = 'Fui al banco a sacar plata'
oracion_parque = 'Me sente en el banco del parque'

v_fin    = embedding_de_palabra(oracion_fin)
v_parque = embedding_de_palabra(oracion_parque)

sim_beto = torch.nn.functional.cosine_similarity(v_fin, v_parque, dim=0).item()
print(f'\nDimension del vector contextual de BETO: {v_fin.shape[0]}')
print(f'\nOracion 1 (financiero): "{oracion_fin}"')
print(f'Oracion 2 (parque):     "{oracion_parque}"')
print(f'\nSimilitud coseno entre los dos "banco": {sim_beto:.3f}')
print('=> Es < 1.0: BETO asigna vectores DISTINTOS a la misma palabra segun el contexto.')
print('   (Con GloVe, del Paso 1, esta similitud era exactamente 1.000.)')


In [ ]:
# @title Paso 3 (visual): GloVe (1.000) vs BETO (<1.0) para el mismo "banco"
fig, ax = plt.subplots(figsize=(7, 4))
etiquetas = ['GloVe (estatico)\nmismo vector', 'BETO (contextual)\ndos contextos']
valores   = [1.000, sim_beto]
barras = ax.bar(etiquetas, valores, color=['#7F8C8D', '#0EA5A0'], alpha=0.9, width=0.55)
for b, v in zip(barras, valores):
    ax.text(b.get_x() + b.get_width()/2, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')
ax.axhline(1.0, color='#E8821A', linestyle='--', linewidth=1, label='similitud = 1.0 (identico)')
ax.set_ylim(0, 1.15)
ax.set_ylabel('Similitud coseno del "banco" en dos contextos')
ax.set_title('Estatico vs. contextual: el salto conceptual', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('Interpretacion:')
print('- GloVe: 1.000  -> imposible distinguir el "banco" financiero del "banco" del parque.')
print(f'- BETO:  {sim_beto:.3f}  -> el vector CAMBIA con la oracion; el modelo capta el contexto.')


In [ ]:
# @title Paso 3 (comprobacion): cada 'banco' se parece mas a su propio contexto
# Comparamos cada 'banco' contextual con una palabra clave de cada sentido:
# 'plata' (financiero) y 'parque' (lugar).
v_plata  = embedding_de_palabra('Fui al banco a sacar plata',      palabra='plata')
v_parq   = embedding_de_palabra('Me sente en el banco del parque', palabra='parque')

def cos(a, b):
    return torch.nn.functional.cosine_similarity(a, b, dim=0).item()

print('Similitud del "banco" de cada oracion con la palabra clave de su contexto:')
print(f'  banco(financiero) ~ plata  : {cos(v_fin, v_plata):.3f}')
print(f'  banco(parque)     ~ parque : {cos(v_parque, v_parq):.3f}')
print()
print('A diferencia de GloVe, aqui el vector de "banco" SI cambia con la oracion,')
print('acercandose al significado correcto en cada caso. Eso es un embedding contextual.')


### Reflexión 3 - Embeddings contextuales

> 1. ¿Por qué la similitud entre los dos "banco" de BETO es < 1.0 mientras que con GloVe era exactamente 1.000?
> 2. ¿De dónde saca BETO la información para diferenciar los dos sentidos si la palabra escrita es idéntica?
> 3. ¿Cómo aprovecharías estos vectores contextuales como *features* para un clasificador de finanzas (en lugar de BoW/TF-IDF del laboratorio anterior)?

# === PAUSA: DISCUSIÓN ===


---
## 🔬 Comparativa multi-modelo: ¿el modelo más grande siempre gana?

Ya usamos un Transformer para sentimiento. Pero en HuggingFace hay **muchos** modelos para la misma
tarea. ¿Cuál elegir? Vamos a poner a competir **tres** modelos de análisis de sentimiento sobre
**los mismos tweets** y medir, de forma objetiva:

- **Tamaño** del modelo (MB en disco) — impacta memoria y costo.
- **Tiempo de inferencia** — impacta latencia y costo en producción.
- **Accuracy y F1** contra una etiqueta "de oro" (`gold`) que definimos nosotros.

Los tres modelos (verificados en HuggingFace):

| Modelo | Idioma / origen | Etiquetas que devuelve |
|---|---|---|
| `pysentimiento/robertuito-sentiment-analysis` | Español (tweets) | POS / NEU / NEG |
| `nlptown/bert-base-multilingual-uncased-sentiment` | Multilingüe (reseñas) | 1 a 5 estrellas |
| `cardiffnlp/twitter-roberta-base-sentiment-latest` | Inglés (tweets) | Negative / Neutral / Positive |

> **Ojo con las etiquetas:** cada modelo devuelve un formato distinto. Necesitamos **normalizar**
> todo a tres clases comunes (`negativo`, `neutro`, `positivo`) para poder compararlos de forma justa.
> El modelo de Cardiff es en **inglés**: lo incluimos a propósito para ver qué pasa cuando el idioma
> del modelo no coincide con el del texto.

In [ ]:
# @title Comparativa (1/3): tweets con etiqueta de oro y normalizacion de etiquetas
# Reutilizamos tweets bancarios colombianos y les asignamos su etiqueta 'gold' (correcta),
# para poder medir accuracy y F1 de forma objetiva.
tweets_eval = [
    ('Bancolombia me acaba de aprobar el credito de vivienda, feliz con la atencion!', 'positivo'),
    ('Llevo 40 minutos en la fila del banco y no avanza nada, pesimo servicio',          'negativo'),
    ('El Banco de la Republica mantuvo la tasa de interes en la reunion de hoy',          'neutro'),
    ('La app de Nequi lleva caida toda la manana, no puedo pagar nada, que rabia',        'negativo'),
    ('Excelente la nueva tarjeta de Davivienda, cashback en todas las compras',           'positivo'),
    ('Segun el DANE la inflacion de este mes fue del 0.3 por ciento',                     'neutro'),
    ('Me cobraron una cuota de manejo que no autorice, voy a reclamar ya mismo',          'negativo'),
    ('Daviplata me salvo el dia, transferencia inmediata y sin filas, lo recomiendo',     'positivo'),
    ('La accion de Ecopetrol cerro estable en la Bolsa de Valores de Colombia',           'neutro'),
    ('Gracias al asesor de Bancolombia que me ayudo a reestructurar la deuda, mil gracias','positivo'),
]
textos = [t for t, _ in tweets_eval]
gold   = [g for _, g in tweets_eval]

# Cada modelo devuelve etiquetas en su propio formato. Las normalizamos a: negativo/neutro/positivo.
def normaliza_etiqueta(label):
    l = str(label).lower()
    # pysentimiento: NEG/NEU/POS
    if l in ('neg', 'negative', 'negativo'):            return 'negativo'
    if l in ('pos', 'positive', 'positivo'):            return 'positivo'
    if l in ('neu', 'neutral', 'neutro'):               return 'neutro'
    # nlptown: '1 star' ... '5 stars'
    if 'star' in l:
        n = int(l.split()[0])
        return 'negativo' if n <= 2 else ('neutro' if n == 3 else 'positivo')
    # cardiff: LABEL_0/1/2 por si acaso
    if l in ('label_0',): return 'negativo'
    if l in ('label_1',): return 'neutro'
    if l in ('label_2',): return 'positivo'
    return l

print('Tweets de evaluacion:', len(textos))
print('Distribucion gold:',
      {c: gold.count(c) for c in ['negativo', 'neutro', 'positivo']})


In [ ]:
# @title Comparativa (2/3): evaluar los 3 modelos (tamaño, tiempo, accuracy, F1)
import time
import numpy as np
import pandas as pd
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score

MODELOS = {
    'RoBERTuito (ES)':      'pysentimiento/robertuito-sentiment-analysis',
    'BERT multiling. (5*)': 'nlptown/bert-base-multilingual-uncased-sentiment',
    'RoBERTa Twitter (EN)': 'cardiffnlp/twitter-roberta-base-sentiment-latest',
}

def tamano_mb(model):
    """Tamaño aproximado del modelo en MB = numero de parametros * 4 bytes (float32)."""
    n_params = sum(p.numel() for p in model.parameters())
    return n_params * 4 / (1024 ** 2)

filas = []
predicciones = {}   # guardamos las predicciones para la discusion posterior

for nombre, model_id in MODELOS.items():
    print(f'Cargando: {nombre}  ({model_id}) ...')
    clf = pipeline('sentiment-analysis', model=model_id, device=DEVICE)

    # Calentamiento (la primera inferencia incluye overhead que no queremos medir)
    _ = clf(textos[0])

    t0 = time.time()
    salidas = clf(textos)                     # inferencia sobre los 10 tweets
    t_infer = time.time() - t0

    pred = [normaliza_etiqueta(s['label']) for s in salidas]
    predicciones[nombre] = pred

    acc = accuracy_score(gold, pred)
    f1  = f1_score(gold, pred, average='macro')
    mb  = tamano_mb(clf.model)

    filas.append({'Modelo': nombre, 'Tamaño (MB)': round(mb, 1),
                  'Tiempo (seg)': round(t_infer, 3),
                  'Accuracy': round(acc, 3), 'F1': round(f1, 3)})
    print(f'  -> {mb:.0f} MB | {t_infer:.3f} s | acc={acc:.2f} | F1={f1:.2f}\n')

tabla = pd.DataFrame(filas).set_index('Modelo')
print('RESULTADOS DE LA COMPARATIVA')
from IPython.display import display
display(tabla)


In [ ]:
# @title Comparativa (3/3): grafico de barras agrupadas (Tamaño, Tiempo, Accuracy, F1)
import matplotlib.pyplot as plt

metricas = ['Tamaño (MB)', 'Tiempo (seg)', 'Accuracy', 'F1']
modelos  = list(tabla.index)
x = np.arange(len(modelos))
ancho = 0.2
colores = ['#1B3A6B', '#0EA5A0', '#16A34A', '#E8821A']

fig, ax = plt.subplots(figsize=(10, 5))
for i, m in enumerate(metricas):
    valores = tabla[m].values.astype(float)
    # Normalizamos cada metrica a [0,1] SOLO para que quepan en el mismo grafico
    # (tamaño y tiempo tienen escalas muy distintas a accuracy/F1).
    v_norm = valores / valores.max() if valores.max() > 0 else valores
    barras = ax.bar(x + (i - 1.5) * ancho, v_norm, ancho, label=m, color=colores[i], alpha=0.9)
    for b, real in zip(barras, valores):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.02,
                f'{real:g}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(modelos, fontsize=9)
ax.set_ylabel('Valor normalizado (0-1) por métrica')
ax.set_title('Comparativa de 3 modelos de sentimiento (etiqueta = valor real)', fontweight='bold')
ax.set_ylim(0, 1.2)
ax.legend(ncol=4, fontsize=8, loc='upper center')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('Nota: las barras estan normalizadas por metrica para poder verlas juntas;')
print('los numeros encima de cada barra son los valores REALES (MB, seg, accuracy, F1).')


### Discusión — ¿el más grande es siempre el mejor?

Observa la tabla y el gráfico:

- El modelo **más grande** (`nlptown`, ~0.2B parámetros, entrenado en reseñas de 1-5 estrellas)
  **no** tiene por qué ganar: su idea de "sentimiento" viene de *reseñas de producto*, no de tweets,
  y su escala de estrellas se pierde información al mapearla a 3 clases.
- El modelo de **Cardiff** es potente, pero está entrenado en **inglés**: sobre tweets en español su
  accuracy suele **caer**. Un modelo más pequeño y **en el idioma correcto** (RoBERTuito) gana con
  frecuencia. **El "match" idioma-dominio importa más que el tamaño.**
- El **tiempo** y el **tamaño** también son parte de la decisión: un modelo que acierta 2 puntos más
  pero es 3x más lento puede **no** valer la pena en producción.

**Conclusión:** elige el modelo por su ajuste a **tu idioma, tu dominio y tus restricciones de
latencia/costo**, no por el número de parámetros.

# === PAUSA: DISCUSIÓN ===

> 1. ¿Cuál modelo dio mejor F1 sobre estos tweets? ¿Coincide con el más grande o con el que está en el idioma/dominio correcto?
> 2. Para un banco que responde tweets en tiempo real, ¿priorizarías 2 puntos más de accuracy o menor latencia? ¿Por qué?
> 3. El modelo multilingüe devuelve 1-5 estrellas. ¿Qué información perdemos al colapsarlo a 3 clases? ¿Cuándo convendría conservar la escala fina?

---
## 👀 Visualización de la atención: ¿qué está "mirando" el Transformer?

En las diapositivas vimos que la **auto-atención (self-attention)** deja que cada palabra "mire" a
las demás para construir su representación contextual. Ahora lo **vemos** con datos reales usando
**BETO** sobre la oración:

> *"El banco central de Colombia subió la tasa de interés"*

Usaremos `bertviz` para una vista interactiva de las cabezas de atención. Si `bertviz` no funciona en
tu entorno de Colab, la siguiente celda tiene un **fallback con matplotlib**: extraemos los pesos de
atención a mano y dibujamos un **heatmap**. Ambos caminos muestran la misma idea.

In [ ]:
# @title Atención (1/2): vista interactiva con bertviz (BETO)
!pip install -q bertviz

import torch
from transformers import AutoTokenizer, AutoModel

BETO = 'dccuchile/bert-base-spanish-wwm-cased'
oracion_attn = 'El banco central de Colombia subio la tasa de interes'

# Cargamos BETO pidiendo explicitamente las matrices de atencion.
tok_attn = AutoTokenizer.from_pretrained(BETO)
mdl_attn = AutoModel.from_pretrained(BETO, output_attentions=True)
mdl_attn.eval()

entrada = tok_attn(oracion_attn, return_tensors='pt')
with torch.no_grad():
    salida_attn = mdl_attn(**entrada)

attention = salida_attn.attentions           # tupla: (n_capas) de (1, n_heads, n_tok, n_tok)
tokens_attn = tok_attn.convert_ids_to_tokens(entrada['input_ids'][0])
print('Tokens:', tokens_attn)
print('Capas:', len(attention), '| cabezas por capa:', attention[0].shape[1])

try:
    from bertviz import head_view
    print('\nMostrando head_view interactivo (elige capa y cabeza en el menu):')
    head_view(attention, tokens_attn)
except Exception as e:
    print('\nbertviz no se pudo mostrar aqui (', type(e).__name__, ').')
    print('No hay problema: ejecuta la siguiente celda para el heatmap con matplotlib.')


In [ ]:
# @title Atención (2/2): fallback con matplotlib — heatmap de una cabeza
import matplotlib.pyplot as plt
import numpy as np

# Elegimos una capa y una cabeza para inspeccionar. En BERT, cabezas de capas
# intermedias suelen capturar relaciones sintacticas (p.ej. adjetivo-sustantivo).
CAPA   = 3     # 0..11
CABEZA = 3     # 0..11

# attention[CAPA]: (1, n_heads, n_tok, n_tok) -> tomamos la cabeza elegida
mat = attention[CAPA][0, CABEZA].detach().cpu().numpy()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(mat, cmap='viridis')
ax.set_xticks(range(len(tokens_attn)))
ax.set_yticks(range(len(tokens_attn)))
ax.set_xticklabels(tokens_attn, rotation=90)
ax.set_yticklabels(tokens_attn)
ax.set_xlabel('Token atendido (hacia donde MIRA)')
ax.set_ylabel('Token que consulta (quien MIRA)')
ax.set_title(f'Pesos de atencion — capa {CAPA}, cabeza {CABEZA}', fontweight='bold')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='peso de atencion')
plt.tight_layout()
plt.show()

# Para cada token, cual es el token al que MAS atiende (fuera de si mismo y de tokens especiales)
print('¿A que palabra atiende mas cada token (capa {}, cabeza {})?'.format(CAPA, CABEZA))
especiales = {'[CLS]', '[SEP]'}
for i, t in enumerate(tokens_attn):
    if t in especiales:
        continue
    fila = mat[i].copy()
    fila[i] = 0                              # ignorar auto-atencion
    j = int(np.argmax(fila))
    print(f'  {t:12s} -> {tokens_attn[j]:12s} (peso {fila[j]:.2f})')


### Guía de lectura — conectar con "Multi-Head Attention"

Mira el `head_view` (o el heatmap): fíjate en cómo, en ciertas cabezas, **"banco"** se conecta con
**"central"**. Esa conexión es exactamente lo que vimos en la diapositiva de **Multi-Head Attention**:
cada cabeza aprende un **tipo de relación distinto** entre palabras (unas capturan relaciones
sintácticas como adjetivo↔sustantivo, otras relaciones de largo alcance).

Prueba a **cambiar `CAPA` y `CABEZA`** en la celda anterior:
- Cabezas de **capas bajas** suelen atender a **tokens vecinos** (posición).
- Cabezas de **capas intermedias/altas** capturan relaciones más **semánticas** ("banco"↔"central",
  "tasa"↔"interés").

Justo por eso "banco central" se interpreta como una **entidad económica** y no como el "banco" del
parque: el modelo **junta** ambas palabras vía atención.

# === PAUSA: DISCUSIÓN ===

> 1. ¿Encuentras una cabeza donde "banco" atienda fuerte a "central"? ¿Qué otras parejas de palabras se conectan?
> 2. ¿Por qué tener **varias** cabezas (multi-head) es mejor que una sola matriz de atención?
> 3. ¿Cómo ayuda esta capacidad de "mirar el contexto" a desambiguar "banco" (lo que vimos con GloVe vs BETO)?

---
## 🧩 Clustering de embeddings: ¿se agrupan los tweets por sentimiento?

Hasta ahora sacamos embeddings de **palabras** sueltas. Ahora vamos a representar **tweets completos**
como un solo vector y a **visualizarlos en 2D** para ver si los tweets con el mismo sentimiento se
agrupan solos, **sin decirle al modelo cuál es el sentimiento**.

Pasos:
1. Un vector por tweet con BETO usando **mean pooling** del `last_hidden_state` (promedio de todos
   los tokens).
2. Reducir de 768 dimensiones a **2D** con **t-SNE** (`perplexity=5`, apropiado para pocos puntos).
3. Graficar cada tweet como un punto **coloreado por su sentimiento** (verde/gris/rojo).

In [ ]:
# @title Clustering (1/2): embeddings de tweet completo con mean pooling (BETO)
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel

# Reutilizamos los mismos tweets del paso de Sentimiento (con su etiqueta de referencia).
tweets_cluster = [
    ('Bancolombia me acaba de aprobar el credito de vivienda, feliz con la atencion!', 'positivo'),
    ('Llevo 40 minutos en la fila del banco y no avanza nada, pesimo servicio',          'negativo'),
    ('El Banco de la Republica mantuvo la tasa de interes en la reunion de hoy',          'neutro'),
    ('La app de Nequi lleva caida toda la manana, no puedo pagar nada, que rabia',        'negativo'),
    ('Excelente la nueva tarjeta de Davivienda, cashback en todas las compras',           'positivo'),
    ('Segun el DANE la inflacion de este mes fue del 0.3 por ciento',                     'neutro'),
    ('Me cobraron una cuota de manejo que no autorice, voy a reclamar ya mismo',          'negativo'),
    ('Daviplata me salvo el dia, transferencia inmediata y sin filas, lo recomiendo',     'positivo'),
    ('La accion de Ecopetrol cerro estable en la Bolsa de Valores de Colombia',           'neutro'),
    ('Gracias al asesor de Bancolombia que me ayudo a reestructurar la deuda, mil gracias','positivo'),
]
textos_c = [t for t, _ in tweets_cluster]
sent_c   = [s for _, s in tweets_cluster]

BETO = 'dccuchile/bert-base-spanish-wwm-cased'
tok_c = AutoTokenizer.from_pretrained(BETO)
mdl_c = AutoModel.from_pretrained(BETO)
mdl_c.eval()
dev_c = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mdl_c.to(dev_c)

def embedding_mean_pooling(texto):
    """Vector del tweet = promedio de los vectores de sus tokens (mean pooling),
    ponderado por la attention_mask para ignorar el padding."""
    enc = tok_c(texto, return_tensors='pt', truncation=True, max_length=64).to(dev_c)
    with torch.no_grad():
        hidden = mdl_c(**enc).last_hidden_state          # (1, n_tok, 768)
    mask = enc['attention_mask'].unsqueeze(-1).float()   # (1, n_tok, 1)
    sumado = (hidden * mask).sum(dim=1)                  # suma solo tokens reales
    conteo = mask.sum(dim=1).clamp(min=1e-9)
    return (sumado / conteo)[0].cpu().numpy()            # (768,)

X = np.vstack([embedding_mean_pooling(t) for t in textos_c])
print('Matriz de embeddings de tweets:', X.shape, '(10 tweets x 768 dimensiones)')


In [ ]:
# @title Clustering (2/2): t-SNE a 2D + scatter coloreado por sentimiento
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# t-SNE a 2D. perplexity=5 porque tenemos MUY pocos puntos (debe ser < n_muestras).
tsne = TSNE(n_components=2, perplexity=5, random_state=42, init='pca', learning_rate='auto')
coords = tsne.fit_transform(X)

COLOR = {'positivo': '#16A34A', 'neutro': '#7F8C8D', 'negativo': '#DC2626'}  # verde/gris/rojo

fig, ax = plt.subplots(figsize=(11, 8))
for clase in ['positivo', 'neutro', 'negativo']:
    idx = [i for i, s in enumerate(sent_c) if s == clase]
    ax.scatter(coords[idx, 0], coords[idx, 1], c=COLOR[clase], s=180,
               label=clase, edgecolors='black', linewidths=0.6, alpha=0.9)

# Anotar cada punto con una version corta del tweet, con un pequeño offset manual.
for i, txt in enumerate(textos_c):
    corto = (txt[:32] + '...') if len(txt) > 32 else txt
    ax.annotate(corto, (coords[i, 0], coords[i, 1]),
                textcoords='offset points', xytext=(8, 6), fontsize=8,
                bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='gray', alpha=0.7))

ax.set_title('Tweets financieros en 2D (t-SNE de embeddings BETO), color = sentimiento',
             fontweight='bold')
ax.set_xlabel('t-SNE dim 1')
ax.set_ylabel('t-SNE dim 2')
ax.legend(title='Sentimiento')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('Nota: t-SNE es estocastico y con 10 puntos la forma varia; fijamos random_state=42')
print('para reproducibilidad. Lo importante es si los colores tienden a agruparse.')


### Reflexión — clustering de embeddings

> 1. ¿Se forman clusters por color (sentimiento) aunque **nunca** le dijimos el sentimiento al modelo?
> 2. ¿Hay algún tweet que quede en la zona "equivocada" (rodeado de otro color)? Léelo: ¿es ambiguo, mezcla emociones o es sarcástico?
> 3. Los tweets **neutros** (noticias, datos del DANE) ¿quedan separados de los de opinión? ¿Por qué tendría sentido que sí?

# === PAUSA: DISCUSIÓN ===

### 🔎 Para los curiosos: mean pooling vs. token [CLS] vs. max pooling

Para convertir los **muchos** vectores de tokens de un tweet en **un solo** vector hay varias
estrategias comunes:

| Estrategia | Cómo funciona | Cuándo conviene |
|---|---|---|
| **Mean pooling** (la que usamos) | Promedia los vectores de todos los tokens (ignorando padding) | Buen valor por defecto; representa el tweet "completo". Es lo que usan modelos tipo *sentence-transformers*. |
| **Token [CLS]** | Toma solo el vector del token especial `[CLS]` del inicio | Rápido; funciona bien **si** el modelo fue afinado para clasificación (el `[CLS]` "resume" la frase). En BERT base sin afinar suele ser peor que mean pooling. |
| **Max pooling** | Toma el máximo por dimensión entre todos los tokens | Resalta rasgos fuertes/salientes (una palabra muy marcada); puede ignorar el resto del contexto. |

> **Idea clave:** el `last_hidden_state` da un vector **por token**; el *pooling* es solo la receta
> para combinarlos en un vector por oración. Para similitud semántica de frases, **mean pooling**
> (o un modelo *sentence-transformers* dedicado) suele ser la opción más robusta.

---
## 🎁 Bonus — Zero-shot classification: clasificar en categorías nunca vistas

> **Este bloque es opcional (bonus).** Si vas corto de tiempo, puedes saltarlo y volver luego.

Todo lo anterior usó modelos ya afinados para una tarea concreta (sentimiento, NER). ¿Y si mañana
el banco necesita clasificar mensajes en categorías **nuevas** (queja, elogio, fraude...) para las que
**no tenemos datos etiquetados**? El **zero-shot classification** permite justamente eso: le pasas el
texto y una lista de **categorías candidatas en texto libre**, y el modelo estima cuál encaja mejor,
**sin haber sido entrenado en esas categorías**.

Usamos **`joeddav/xlm-roberta-large-xnli`**, un modelo multilingüe (soporta español) entrenado en
**NLI (Natural Language Inference)**.

In [ ]:
# @title Bonus: zero-shot classification con categorias nuevas (nunca entrenadas)
from transformers import pipeline
import pandas as pd
from IPython.display import display

# Es un modelo grande (~0.6B); en Colab conviene la GPU T4 (device=DEVICE).
zsc = pipeline('zero-shot-classification',
               model='joeddav/xlm-roberta-large-xnli',
               device=DEVICE)

# 5 tweets financieros colombianos.
tweets_zs = [
    'Llevo tres dias esperando que me devuelvan el dinero de una compra que no reconozco',
    'La verdad la nueva tarjeta de Davivienda es una maravilla, la recomiendo',
    'A que hora abren hoy la sucursal de Bancolombia del centro?',
    'Quisiera saber los requisitos para un credito de libre inversion',
    'Me llego un mensaje raro pidiendo mi clave, creo que es un intento de estafa',
]

# Categorias que el modelo NUNCA vio como clases de entrenamiento.
categorias = ['queja de servicio', 'elogio al producto', 'consulta de informacion',
              'reporte de fraude', 'solicitud de credito']

# En español conviene una plantilla de hipotesis en español (recomendado por el modelo).
PLANTILLA = 'Este texto es una {}.'

filas = []
for tw in tweets_zs:
    r = zsc(tw, candidate_labels=categorias, hypothesis_template=PLANTILLA)
    # r['labels'] y r['scores'] vienen ordenados de mayor a menor.
    fila = {'tweet': (tw[:55] + '...') if len(tw) > 55 else tw,
            'prediccion': r['labels'][0]}
    for etiqueta, score in zip(r['labels'], r['scores']):
        fila[etiqueta] = round(score, 3)
    filas.append(fila)

# Tabla: columnas de categorias en orden fijo + la prediccion top.
tabla_zs = pd.DataFrame(filas).set_index('tweet')
tabla_zs = tabla_zs[['prediccion'] + categorias]

print('ZERO-SHOT: probabilidad por categoria (categorias nunca entrenadas)')
# Resaltamos la categoria ganadora de cada fila.
try:
    display(tabla_zs.style.background_gradient(cmap='Greens', subset=categorias)
            .format({c: '{:.3f}' for c in categorias}))
except Exception:
    display(tabla_zs)


### Reflexión — ¿cómo clasifica en categorías que nunca entrenó?

El truco es que el modelo **no** aprendió estas categorías: fue entrenado en **NLI (Natural Language
Inference)**, la tarea de decidir si una hipótesis se **deduce** (*entailment*), **contradice** o es
**neutral** respecto a un texto (premisa). El pipeline convierte la clasificación en NLI así:

- **Premisa** = el tweet.
- **Hipótesis** = *"Este texto es una queja de servicio."* (una por cada categoría candidata).
- La categoría con mayor probabilidad de **entailment** gana.

Por eso puedes inventar **categorías nuevas** sobre la marcha, en español, sin reentrenar nada.

# === PAUSA: DISCUSIÓN ===

> 1. ¿Acertó el modelo en los 5 tweets? ¿En cuál dudó más (probabilidades repartidas entre 2 categorías)?
> 2. Como es zero-shot, puedes cambiar las categorías sin reentrenar. ¿Qué categorías propondrías para el canal de mensajes de un banco?
> 3. ¿Qué ventaja tiene esto frente a entrenar un clasificador supervisado? ¿Qué desventajas (costo, tamaño del modelo, precisión) esperarías?

---
## 🤔 Paso 4 — ¿Cuándo Vale la Pena un Transformer vs. BoW?

Después de ver el poder de los Transformers, es tentador pensar que **siempre** son la mejor opción.
Pero en la práctica **no siempre el modelo más sofisticado es el mejor**: un Transformer es más
lento, necesita GPU, es más difícil de explicar y toma más tiempo de desarrollo.

La pregunta correcta no es *"¿cuál es más potente?"*, sino *"¿qué necesita realmente el problema?"*.
Muchas tareas de negocio se resuelven perfectamente con **BoW/TF-IDF + Regresión Logística** (el lab de clasificación con BoW),
que es rápido, barato e interpretable. Comparemos ambos enfoques lado a lado para decidir con criterio.


In [ ]:
# @title Paso 4: Tabla comparativa BoW/TF-IDF vs Transformer
import pandas as pd
from IPython.display import display

# 🟢 = ventaja  |  🟡 = neutro/depende  |  🔴 = desventaja
comparacion = pd.DataFrame(
    {
        'BoW/TF-IDF (clasico)': [
            '🟢 Muy rapida (ms, en CPU)',
            '🟢 No requiere GPU',
            '🟢 Suele bastar (0.80-0.90)',
            '🔴 Falla con negacion/sarcasmo',
            '🟡 Necesita datos etiquetados',
            '🟢 Alta (ves que palabras pesan)',
            '🟢 Rapido de prototipar',
        ],
        'Transformer (este lab)': [
            '🔴 Mas lenta (mejor con GPU)',
            '🔴 Ideal/necesita GPU',
            '🟡 Bueno, pero puede ser overkill',
            '🟢 Entiende contexto y negacion',
            '🟢 Funciona pre-entrenado (0 datos)',
            '🔴 Baja (caja negra)',
            '🟡 Mas setup, pero pipelines ayudan',
        ],
    },
    index=[
        'Velocidad de inferencia',
        'Costo de GPU',
        'Precision en tareas simples',
        'Precision en tareas complejas (negacion, sarcasmo)',
        'Necesidad de datos etiquetados',
        'Interpretabilidad',
        'Tiempo de desarrollo',
    ],
)
comparacion.index.name = 'Criterio'

print('COMPARATIVA: modelo clasico vs. Transformer')
print('Leyenda: 🟢 ventaja   🟡 neutro/depende   🔴 desventaja\n')
display(comparacion)


### 🧭 Guía de decisión rápida

Un pequeño "árbol de decisión" en texto para elegir enfoque:

```
┌─ ¿Tienes < 1000 ejemplos etiquetados?
│     └─ SI  → BoW/TF-IDF puede ser suficiente (y evitas sobre-ingenieria)
│
├─ ¿El texto tiene negaciones, sarcasmo o jerga?
│     └─ SI  → Transformer (capta contexto; BoW se queda corto)
│
├─ ¿Necesitas EXPLICAR cada prediccion (auditoria, regulacion)?
│     └─ SI  → BoW/TF-IDF (mas interpretable; ves que palabras pesan)
│
├─ ¿Tienes GPU disponible en produccion?
│     └─ NO  → BoW/TF-IDF (corre barato en CPU)
│     └─ SI  → Transformer es viable
│
└─ ¿Es un MVP / prototipo para YA?
      └─ SI  → BoW primero (rapido y barato), Transformer despues si hace falta
```

#### 💡 Ejemplo concreto (contexto bancario colombiano)

> Si **Bancolombia** quiere clasificar **10,000 tweets** de quejas vs. elogios y necesita un
> **prototipo para mañana**, empieza con **TF-IDF + Regresión Logística**: entrena en segundos,
> corre en CPU y es fácil de explicar al equipo de negocio.
>
> Si **después** necesita detectar **sarcasmo** del tipo *"Ay sí, excelente servicio 🙄"* —donde las
> palabras son positivas pero la intención es negativa— **ahí sí justifica un Transformer**, porque
> BoW clasificaría ese tweet como "positivo" al ver "excelente" y "servicio".

**Moraleja:** empieza simple, mide, y sube de complejidad **solo cuando los datos lo justifiquen**.


### Reflexión 4 - Elegir la herramienta correcta

# === PAUSA: DISCUSIÓN ===

> 1. ¿En tu empresa, qué tipo de textos procesan? ¿Serían candidatos a Transformer o BoW sería suficiente?
> 2. ¿Cuál es el costo real de usar GPU en producción vs. el valor que aporta la mejora en precisión?
> 3. ¿Qué pasa si el 95% de los casos se resuelve con BoW y solo el 5% necesita Transformer? ¿Cómo diseñarías un sistema que combine ambos (p. ej. BoW por defecto y Transformer solo para los casos dudosos)?


---
## Reflexiones finales

# === PAUSA: DISCUSIÓN FINAL ===

> **1. Estático vs. contextual:** con tus palabras, ¿cuál es la diferencia entre el vector de GloVe
> para "banco" y el vector que produce BETO para "banco" en dos oraciones distintas?
>
> **2. Pipelines:** ¿qué tres pasos encapsula un pipeline de HuggingFace? ¿Por qué eso acelera tanto
> el desarrollo?
>
> **3. Cuándo usar qué:** para un banco colombiano que recibe 5.000 tweets/día y necesita saber
> *por qué* se clasifica algo como queja, ¿usarías el Transformer, el modelo clásico de BoW, o una
> combinación?
>
> **4. Siguiente paso:** hoy solo hicimos *inferencia*. ¿Qué haría falta para *fine-tunear* uno de
> estos modelos con los datos etiquetados de TU propio banco?

---

## Resumen del Lab

| Concepto | Aprendizaje clave |
|---|---|
| **Embedding estático (GloVe)** | Un vector fijo por palabra: "banco" vs "banco" = similitud 1.000, no distingue sentidos |
| **Embedding contextual (BETO)** | Un vector por aparición: "banco" cambia según el contexto (similitud < 1.0) |
| **HuggingFace** | Repositorio público de modelos pre-entrenados ("GitHub de la IA") |
| **pipeline()** | Tokeniza + infiere + formatea en una sola llamada; `device=DEVICE` usa la GPU T4 |
| **Sentiment analysis** | Clasifica sentimiento sin entrenar; supera a BoW en negación e informalidad |
| **NER** | Extrae entidades (PER/ORG/LOC/MISC) de noticias financieras: imposible con Bag of Words |

### Modelos usados (verificados en HuggingFace)
- `glove-wiki-gigaword-100` (vía `gensim`)
- `pysentimiento/robertuito-sentiment-analysis`
- `mrm8488/bert-spanish-cased-finetuned-ner`
- `dccuchile/bert-base-spanish-wwm-cased` (BETO)
